# 01 — Trace one control step

## Goal
Predict what changes when you write one actuator target. Then inspect the state before and after physics steps.

This is the first entry in [the learning route](../LEARNING.md). No RL or PPO implementation is supplied. This notebook uses the canonical simulation core. It does not start a viewer, socket server, or controller.

**Run All is setup-only by default.** Write your prediction before you enable the experiment. No measured results are saved in this notebook.

## Setup
From the `spider` repository root, install `requirements-learning.txt` and start `python -m jupyter lab`. Use that Python environment as the kernel. This notebook supports a kernel working directory at the repo root or in `notebooks/`.

The simulation dependencies are pinned in `requirements.txt`. The setup below prints installed package versions. Record those versions with any later comparison.

In [ ]:
from pathlib import Path
from importlib.metadata import version
import sys

candidates = (Path.cwd(), Path.cwd().parent)
REPO_ROOT = next(
    (path for path in candidates if (path / "simulation.py").is_file()
     and (path / "model" / "spider.xml").is_file()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Start the kernel in spider/ or spider/notebooks/.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import mujoco
import numpy as np
import matplotlib.pyplot as plt
from simulation import load_model, measured_state, neutral_targets, reset, set_targets, step

print("Repository:", REPO_ROOT)
print("Python:", sys.version.split()[0])
print({name: version(name) for name in ("mujoco", "numpy", "matplotlib")})

## Steps
### 1. Make an Iteration 0 prediction
Use the physical picture first. Gravity acts on the robot. Ground contacts apply forces to its feet. These forces can create torques about the leg joints. Locate the actuator that can oppose or change the selected joint torque.

Without running the experiment, predict:
- Which of time, command, joint position, joint velocity, and actuator force change immediately after the target write?
- Which can change after one physics step? Which can change over several steps?
- What would make your prediction wrong?

Write your reasoning below. Leave uncertainty visible. You do not need a formal mechanics derivation to start.

In [ ]:
PREDICTION = ""  # Write your prediction and reason here.
RUN_EXPERIMENT = False  # Set True only after writing your prediction.

ACTUATOR_NAME = "front_left_hip_motor"
TARGET_OFFSET_RAD = 0.05
PHYSICS_STEPS = 100  # Observation window, not a policy-rate decision.

### 2. Prepare raw state capture
These helpers only define setup. The experiment creates a fresh model and state each time it runs. It resolves the selected joint through the actuator transmission instead of assuming an index.

Angles and targets use radians. Angular velocity uses radians per second. Time uses seconds. The selected hinge actuator force uses newton metres. The raw arrays retain MuJoCo's sampling order. Do not treat every derived field as a freshly recomputed post-step value without checking that order in the core.

In [ ]:
def capture_raw(model, data, actuator_id, joint_id, label):
    position_address = model.jnt_qposadr[joint_id]
    velocity_address = model.jnt_dofadr[joint_id]
    return {
        "label": label,
        "time_s": float(data.time),
        "target_rad": float(data.ctrl[actuator_id]),
        "joint_position_rad": float(data.qpos[position_address]),
        "joint_velocity_rad_s": float(data.qvel[velocity_address]),
        "actuator_torque_nm": float(data.actuator_force[actuator_id]),
    }


def run_target_experiment():
    if not isinstance(PHYSICS_STEPS, int) or PHYSICS_STEPS < 1:
        raise ValueError("PHYSICS_STEPS must be a positive integer.")
    model = load_model()
    data = mujoco.MjData(model)
    reset(model, data)
    actuator_id = model.actuator(ACTUATOR_NAME).id
    joint_id = int(model.actuator_trnid[actuator_id, 0])
    targets = np.array(neutral_targets(), dtype=float)
    targets[actuator_id] += TARGET_OFFSET_RAD
    lower, upper = model.actuator_ctrlrange[actuator_id]
    if not np.isfinite(targets[actuator_id]) or not lower <= targets[actuator_id] <= upper:
        raise ValueError("Choose a finite target inside the actuator control range.")
    snapshots = [capture_raw(model, data, actuator_id, joint_id, "reset")]
    set_targets(data, targets)
    snapshots.append(capture_raw(model, data, actuator_id, joint_id, "target written"))
    for index in range(PHYSICS_STEPS):
        step(model, data)
        snapshots.append(capture_raw(model, data, actuator_id, joint_id, f"step {index + 1}"))
    return snapshots, measured_state(model, data), float(model.opt.timestep)

### 3. Inspect only after predicting
Enable the gate above, then run this cell and the plot cell. Start with the first three records. Compare your prediction with the raw values before interpreting the longer trace.

In [ ]:
experiment_ran = False
snapshots = None
final_state = None
if RUN_EXPERIMENT and PREDICTION.strip():
    snapshots, final_state, timestep_s = run_target_experiment()
    experiment_ran = True
    print("Physics timestep (s):", timestep_s)
    for snapshot in snapshots[:3]:
        print(snapshot)
else:
    print("Experiment paused. Write PREDICTION and set RUN_EXPERIMENT=True to inspect results.")

### 4. Compare the command and measured angle
The plot shows the selected joint only. It does not establish whole-body stability or explain causality. Use the raw records to distinguish the target write from the first physics step.

In [ ]:
if RUN_EXPERIMENT and PREDICTION.strip() and experiment_ran:
    times = [row["time_s"] for row in snapshots]
    fig, axis = plt.subplots(figsize=(7, 3))
    axis.plot(times, [row["target_rad"] for row in snapshots], label="Command target")
    axis.plot(times, [row["joint_position_rad"] for row in snapshots], label="Measured joint angle")
    axis.set(xlabel="Simulation time (s)", ylabel="Angle (rad)", title=ACTUATOR_NAME)
    axis.legend()
    axis.grid(alpha=0.25)
    fig.tight_layout()
    plt.show()

## Checks
Write your own interpretation after inspecting the records.

1. Cite one observation that supports or refutes your prediction.
2. Trace the target write and physics step in `simulation.py`. Locate the corresponding joint and actuator in `model/spider.xml`.
3. Explain which evidence you still need to connect target, torque, motion, gravity, and ground contact.
4. Predict a second offset or a different joint before changing the parameters. Use that independent attempt to check transfer.

Do not mark this learning step complete because Run All succeeds.

In [ ]:
OBSERVATION = ""
REVISED_EXPLANATION = ""
NEXT_PREDICTION = ""

## Next Steps
Read [LEARNING.md](../LEARNING.md) with your pair programmer. Define observations, reward components, failure conditions, and policy timing together before training.

`learning_env.LearningSimulation` provides `reset()` and `step(target_offsets_rad, physics_steps=...)`. It accepts offsets around the neutral targets and returns `MeasuredState`. It is a physics adapter, not a Gym environment or a trainer. It does not choose reward, termination, or policy timing for you.

You will write the RL loop and PPO with PyTorch. Keep the first explanation and implementation attempt yours.